# Paper 3 — DICHASUS cf0x: Preprocessing, Reference Statistics, Predictability

Three jobs in one pass over the raw TFRecords:

1. **Preprocessing for adaptive filtering.** The WARDEN pipeline subsamples time by `every_n=5`,
   which is harmless for channel charting but fatal here: at 1.272 GHz (lambda = 23.6 cm) and
   20.83 Hz native rate, every 5th sample means the transmitter moves up to half a wavelength
   between samples and the CSI phase decorrelates — the same failure mode that gave NMSE 2.79 in the
   first M2 run. We keep **every** sample and decimate in frequency instead (1024 -> 64 subcarriers,
   matching the simulation).
2. **Reference statistics** (RMS delay spread, K-factor per array) computed at the full 1024-subcarrier
   resolution while streaming, saved as `dichasus_ref_stats.json` for the M1 sim-to-real gate.
3. **Predictability check**: per-sample displacement against lambda, and the naive-predictor NMSE on
   real CSI. This decides whether E0/E1 transfer to measured data at all.

cf0x: 4 arrays x 8 antennas, 1024 subcarriers, 50 MHz, 1.272 GHz, factory hall, robot-mounted
transmitter on meandering trajectories. Start with cf02 (smallest file); cf03/cf04 can be added.

In [ ]:
# %% [1] Setup
!pip install -q tensorflow 2>&1 | tail -1
import os, json, uuid, glob, datetime
import numpy as np, tensorflow as tf
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
WARDEN = '/content/drive/MyDrive/WH_RDCC'
RAW = os.path.join(WARDEN, 'data_raw')
DICH_DIR = os.path.join(BASE, 'dichasus'); os.makedirs(DICH_DIR, exist_ok=True)

def drive_canary(path=DICH_DIR):
    token = str(uuid.uuid4()); c = os.path.join(path, '.canary')
    with open(c, 'w') as f: f.write(token)
    with open(c) as f: assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount.'
    os.remove(c); print('Drive canary OK:', path)

drive_canary()
print(sorted(os.listdir(RAW)))

In [ ]:
# %% [2] Configuration
FILES = ['cf02']                 # add 'cf03', 'cf04' once cf02 is validated
N_ANT = 32
N_SUB = 1024
N_ARRAY = 4                      # 4 arrays x 8 antennas; ordering verified in cell [3]
SUB_DEC = 16                     # 1024 -> 64 subcarriers for the filtering experiments
# BW/FC and the antenna->array map are read from spec.json in cell [3], not assumed.
MAX_GAP_S = 0.15                 # a larger gap starts a new trajectory segment
CHUNK = 2000                     # samples per checkpoint

In [ ]:
# %% [3] Inspect spec.json — antenna-to-array mapping must be verified, not assumed
with open(os.path.join(RAW, 'spec.json')) as f:
    spec = json.load(f)
BW_HZ, FC_HZ = float(spec['bandwidth']), float(spec['fc'])
LAMBDA = 3e8/FC_HZ

# spec.json lists the arrays in an order that does NOT match antenna indices: its first entry owns
# antennas 24-31. Derive the map instead of assuming it - the grouping matters for statistics, the
# ordering for array geometry (E3).
ANT2ARR = np.zeros(N_ANT, np.int64)
ARRAY_POS = np.zeros((N_ARRAY, 3), np.float64)
for entry in spec['antennas']:
    ants = sorted(i for row in entry['assignments'] for i in row)
    a = ants[0]//(N_ANT//N_ARRAY)
    ANT2ARR[ants] = a
    ARRAY_POS[a] = entry['location']
ARR_IDX = [np.where(ANT2ARR == a)[0] for a in range(N_ARRAY)]
print(f'lambda = {LAMBDA*100:.1f} cm | delay resolution = {1/BW_HZ*1e9:.1f} ns | fc {FC_HZ/1e9:.3f} GHz')
for a in range(N_ARRAY):
    print(f'  array {a}: antennas {ARR_IDX[a].min()}-{ARR_IDX[a].max()} at {ARRAY_POS[a][:2]}')
np.save(os.path.join(DICH_DIR, 'array_positions.npy'), ARRAY_POS)

In [ ]:
# %% [4] Streaming preprocessing: no temporal subsampling, full-resolution channel statistics
def parse_fn(proto):
    rec = tf.io.parse_single_example(proto, {
        'csi': tf.io.FixedLenFeature([], tf.string, default_value=''),
        'pos-tachy': tf.io.FixedLenFeature([], tf.string, default_value=''),
        'time': tf.io.FixedLenFeature([], tf.float32, default_value=0)})
    csi = tf.ensure_shape(tf.io.parse_tensor(rec['csi'], out_type=tf.float32), (N_ANT, N_SUB, 2))
    csi = tf.complex(csi[:, :, 0], csi[:, :, 1])
    csi = tf.signal.fftshift(csi, axes=1)
    pos = tf.ensure_shape(tf.io.parse_tensor(rec['pos-tachy'], out_type=tf.float64), (3,))
    return csi, pos, rec['time']

def calibrate(csi, offsets):
    k = np.arange(N_SUB, dtype=np.float32)
    sto = np.outer(np.array(offsets['sto'], np.float32), 2*np.pi*k/N_SUB)
    cpo = np.outer(np.array(offsets['cpo'], np.float32), np.ones(N_SUB, np.float32))
    return csi*np.exp(1j*(sto + cpo)).astype(np.complex64)

def channel_stats(csi):
    """Per-array RMS delay spread and K-factor from the full 1024-subcarrier CSI."""
    h = np.fft.ifft(np.fft.ifftshift(csi, axes=-1), axis=-1)          # [32, 1024] time domain
    p = np.abs(h)**2
    tau = np.arange(N_SUB)/BW_HZ
    ds, kf, en = np.zeros(N_ARRAY), np.zeros(N_ARRAY), np.zeros(N_ARRAY)
    for a, idx in enumerate(ARR_IDX):
        pa = p[idx].mean(0)                                            # average over the array
        s = pa.sum() + 1e-30
        m = (pa*tau).sum()/s
        ds[a] = np.sqrt((pa*(tau-m)**2).sum()/s)
        kf[a] = 10*np.log10(pa.max()/max(s - pa.max(), 1e-30))
        en[a] = 10*np.log10((np.abs(csi[idx])**2).sum() + 1e-30)
    return ds, kf, en

for fid in FILES:
    out_npz = os.path.join(DICH_DIR, f'dichasus_{fid}_full.npz')
    if os.path.exists(out_npz):
        print(f'{fid}: already preprocessed, skipping'); continue
    with open(os.path.join(RAW, f'reftx-offsets-dichasus-{fid}.json')) as f:
        offsets = json.load(f)
    ds_ = tf.data.TFRecordDataset(os.path.join(RAW, f'dichasus-{fid}.tfrecords'))
    ds_ = ds_.map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE).prefetch(8)

    csi_l, pos_l, t_l, ds_l, kf_l, en_l = [], [], [], [], [], []
    for i, (csi, pos, t) in enumerate(ds_):
        c = calibrate(csi.numpy(), offsets)
        d, k, e = channel_stats(c)
        csi_l.append(c[:, ::SUB_DEC].astype(np.complex64))            # [32, 64]
        pos_l.append(pos.numpy()[:2]); t_l.append(float(t.numpy()))
        ds_l.append(d); kf_l.append(k); en_l.append(e)
        if (i+1) % CHUNK == 0:
            print(f'  {fid}: {i+1} samples')
    drive_canary()
    np.savez_compressed(out_npz, csi=np.stack(csi_l), pos=np.stack(pos_l).astype(np.float32),
                        time=np.array(t_l), ds=np.stack(ds_l).astype(np.float32),
                        kf=np.stack(kf_l).astype(np.float32), en=np.stack(en_l).astype(np.float32))
    print(f'{fid}: saved {len(csi_l)} samples -> {out_npz} '
          f'({os.path.getsize(out_npz)/1e6:.0f} MB)')

In [ ]:
# %% [5] Trajectory segments, sampling rate, and displacement against lambda
Z = np.load(os.path.join(DICH_DIR, f'dichasus_{FILES[0]}_full.npz'))
csi, pos, tm = Z['csi'], Z['pos'], Z['time']
order = np.argsort(tm); csi, pos, tm = csi[order], pos[order], tm[order]
dt = np.diff(tm)
seg_break = np.where(dt > MAX_GAP_S)[0]
segs = np.split(np.arange(len(tm)), seg_break + 1)
segs = [s for s in segs if len(s) >= 200]
step = np.linalg.norm(np.diff(pos, axis=0), axis=1)
inside = np.ones(len(step), bool); inside[seg_break] = False

print(f'{len(tm)} samples | median dt {np.median(dt[inside])*1000:.1f} ms '
      f'({1/np.median(dt[inside]):.1f} Hz) | {len(segs)} segments >=200 samples '
      f'(longest {max(len(s) for s in segs)})')
print(f'step per sample: median {np.median(step[inside])*100:.1f} cm = '
      f'{np.median(step[inside])/LAMBDA:.3f} lambda | P90 {np.percentile(step[inside], 90)/LAMBDA:.3f} lambda')
print('[CHECK] median step < 0.15 lambda (phase-coherent, filterable):',
      'PASS' if np.median(step[inside])/LAMBDA < 0.15 else 'FAIL - subsample-free data still too coarse')

In [ ]:
# %% [6] Reference statistics for the M1 sim-to-real gate
ds_ns = Z['ds'][:, :]*1e9
en = Z['en']
serving = en.argmax(1)
ds_serv = np.take_along_axis(ds_ns, serving[:, None], 1)[:, 0]
kf_serv = np.take_along_axis(Z['kf'], serving[:, None], 1)[:, 0]
ref = dict(dataset=FILES[0], n_samples=int(len(ds_serv)),
           ds_ns=ds_serv[::10].tolist(),                    # decimated sample for the gate CDF
           ds_median_ns=float(np.median(ds_serv)), ds_p90_ns=float(np.percentile(ds_serv, 90)),
           kf_median_dB=float(np.median(kf_serv)),
           carrier_hz=FC_HZ, bandwidth_hz=BW_HZ)
drive_canary()
with open(os.path.join(BASE, 'dichasus_ref_stats.json'), 'w') as f:
    json.dump(ref, f)
print(f"serving-link RMS delay spread: median {ref['ds_median_ns']:.0f} ns | "
      f"P90 {ref['ds_p90_ns']:.0f} ns | K-factor median {ref['kf_median_dB']:.1f} dB")
print('Saved dichasus_ref_stats.json — the M1 gate cell will pick it up automatically.')

plt.figure(figsize=(5, 4))
plt.hist(ds_serv, bins=80, cumulative=True, density=True, histtype='step')
plt.xlabel('RMS delay spread [ns]'); plt.ylabel('CDF'); plt.grid(alpha=0.3)
plt.title(f'DICHASUS {FILES[0]} serving link'); plt.show()

In [ ]:
# %% [7] Predictability of real CSI: naive predictor per segment
# Same metric as M2/E0: NMSE of h_t = h_{t-1} on the serving array's CSI vector.
seg = max(segs, key=len)
X = csi[seg]                                                  # [T, 32, 64]
en_seg = Z['en'][seg]
a = np.bincount(en_seg.argmax(1), minlength=N_ARRAY).argmax()  # dominant array over the segment
hv = X[:, ARR_IDX[a]].reshape(len(seg), -1)
hv = hv/(np.sqrt((np.abs(hv)**2).mean()) + 1e-30)
nrm = np.linalg.norm(hv, axis=1) + 1e-30
naive = (np.linalg.norm(hv[1:]-hv[:-1], axis=1)**2/nrm[1:]**2).mean()
rho_c = [np.abs(((hv[:-L or None].conj()*hv[L:]).sum(1)/(nrm[:-L or None]*nrm[L:])).mean())
         if L else 1.0 for L in range(21)]
print(f'longest segment: {len(seg)} samples, array {a} | naive-predictor NMSE {naive:.4f}')
print('(simulation reference: 0.126 static / 0.171 dynamic at 5 ms)')
plt.figure(figsize=(5, 4))
plt.plot(np.arange(21)*np.median(dt[inside])*1000, rho_c, 'o-')
plt.xlabel('lag [ms]'); plt.ylabel('coherent |E[rho]|'); plt.grid(alpha=0.3)
plt.title('Real CSI temporal coherence'); plt.show()

In [ ]:
# %% [8] Full-dataset (tau, phi) alignment — jointly over ALL 32 antennas, per segment
# The jitter is one (delay, phase) per snapshot, common to the whole capture (single sounder clock),
# so it is estimated on all antennas jointly. Inter-array relative ToAs within a snapshot are
# untouched: the E3 TDoA features survive. Alignment is causal (each snapshot to the previous
# aligned one), i.e. exactly what a UE's timing/frequency loops would do.
def align_segment(Xs):
    T, A_, S = Xs.shape
    k = np.arange(S)
    out = np.empty_like(Xs); out[0] = Xs[0]
    for t in range(1, T):
        x = (out[t-1].conj()*Xs[t]).sum(0)                    # [S] correlation vs aligned previous
        w = np.abs(x[1:]*x[:-1])
        slope = np.angle(np.sum(w*np.exp(1j*np.angle(x[1:]*x[:-1].conj()))))
        corr = np.exp(-1j*slope*k)
        phi = np.angle((x*corr).sum())
        out[t] = Xs[t]*(corr*np.exp(-1j*phi))[None, :]
    return out

aligned_path = os.path.join(DICH_DIR, f'dichasus_{FILES[0]}_aligned.npz')
if os.path.exists(aligned_path):
    print('already aligned, loading'); ZA = np.load(aligned_path)
    csi_al, seg_id = ZA['csi'], ZA['seg_id']
else:
    csi_al = csi.copy()
    seg_id = np.full(len(csi), -1, np.int32)
    for si, s in enumerate(segs):
        csi_al[s] = align_segment(csi[s])
        seg_id[s] = si
        print(f'segment {si}: {len(s)} samples aligned')
    drive_canary()
    np.savez_compressed(aligned_path, csi=csi_al, pos=pos, time=tm, seg_id=seg_id, en=Z['en'])
    print('Saved:', aligned_path)

In [ ]:
# %% [9] E0 on real data: naive / fixed Wiener / NLMS / RLS on aligned segments
P_E0, MU_NLMS, RLS_LAMBDA, RLS_DELTA, RCOND, TRAIN_FRAC = 4, 0.5, 0.995, 1e-2, 1e-4, 0.5

def lagm(X, p, t): return X[t-p:t][::-1]

def run_pred(X, p, algo):
    T, K = X.shape
    t_split = int(TRAIN_FRAC*T)
    w = np.zeros((p, K), np.complex128)
    if algo == 'wiener':
        Xtr = X[:t_split]
        r = np.zeros((p+1, K), np.complex128); r[0] = (np.abs(Xtr)**2).mean(0)
        for kk in range(1, p+1): r[kk] = (Xtr[kk:]*np.conj(Xtr[:-kk])).mean(0)
        R = np.zeros((K, p, p), np.complex128)
        for i in range(p):
            for j in range(p):
                R[:, i, j] = np.conj(r[i-j]) if i >= j else r[j-i]
        w = (np.linalg.pinv(R, rcond=RCOND) @ np.conj(r[1:p+1]).T[..., None])[..., 0].T
    if algo == 'rls':
        Pm = np.tile((1.0/RLS_DELTA)*np.eye(p, dtype=np.complex128), (K, 1, 1))
    e_all, d_all = [], []
    for t in range(p, T):
        u = lagm(X, p, t).astype(np.complex128)
        if algo == 'naive':
            e = X[t] - X[t-1]
        else:
            e = X[t] - (np.conj(w)*u).sum(0)
        e_all.append(e); d_all.append(X[t])
        if algo == 'nlms':
            w += MU_NLMS*u*np.conj(e)/((np.abs(u)**2).sum(0) + 1e-6)
        elif algo == 'rls':
            Pu = np.einsum('kij,jk->ik', Pm, u)
            den = RLS_LAMBDA + (np.conj(u)*Pu).sum(0)
            g = Pu/den; w += g*np.conj(e)
            uHP = np.einsum('ik,kij->jk', np.conj(u), Pm)
            Pm = (Pm - np.einsum('ik,jk->kij', g, uHP))/RLS_LAMBDA
    e_all, d_all = np.array(e_all), np.array(d_all)
    m = np.zeros(len(e_all), bool); m[t_split-p:] = True
    return float((np.abs(e_all[m])**2).sum()/((np.abs(d_all[m])**2).sum() + 1e-30))

EN = Z['en']
res = {a_: [] for a_ in ('naive', 'wiener', 'nlms', 'rls')}
weights = []
for si, s in enumerate(segs):
    if len(s) < 500: continue
    arr = np.bincount(EN[s].argmax(1), minlength=N_ARRAY).argmax()
    Xs = csi_al[s][:, ARR_IDX[arr]].reshape(len(s), -1)
    Xs = Xs/(np.sqrt((np.abs(Xs)**2).mean()) + 1e-30)
    row = f'seg {si:2d} ({len(s):4d} smp, array {arr})'
    for algo in res:
        nm = run_pred(Xs, P_E0, algo)
        res[algo].append(nm); row += f' | {algo} {nm:.4f}'
    weights.append(len(s)); print(row)

w_ = np.array(weights, float); w_ /= w_.sum()
print()
print('Weighted over segments (>=500 samples):')
for algo, v in res.items():
    print(f'  {algo:7s} NMSE {np.average(v, weights=w_):.4f}')
print('Simulation reference (static walk, p=4): naive 0.1265 | wiener 0.0018 | nlms 0.0018 | rls 0.0009')